# Claude API Tool Calling — Under the Hood

A hands-on Python lab showing how an LLM can choose Python functions, pass arguments, receive results, and continue the agent loop.

**Goal:** understand the mechanism before introducing MCP or an agent framework.


## 0. Setup

Install the SDK and load `ANTHROPIC_API_KEY`. This notebook uses the direct Messages API so the tool-calling loop is visible.

In [ ]:
!pip install -q anthropic python-dotenv

import os
import json
from dotenv import load_dotenv
import anthropic

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

client = anthropic.Anthropic()
MODEL = "claude-opus-5-5"
print("Claude client ready")


## 1. First: a normal Claude call

Nothing agentic yet. We send a message and Claude returns text.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explain MLOps in three sentences."}]
)

print(response.content[0].text)
print("\nStop reason:", response.stop_reason)


## 2. Create ordinary Python functions

These are the functions Claude will eventually be allowed to request. Claude does **not** execute them. Your Python program does.

In [ ]:
def get_model_metrics(model_name: str):
    models = {
        "fraud-model": {"accuracy": 0.94, "latency_ms": 82, "version": "17"},
        "recommendation-model": {"accuracy": 0.91, "latency_ms": 145, "version": "8"},
    }
    return models.get(model_name, {"error": "Model not found"})


def get_kubernetes_status(model_name: str):
    return {
        "model": model_name,
        "replicas": 3,
        "ready": 3,
        "status": "Running",
    }


def get_recent_deployments(model_name: str):
    return {
        "model": model_name,
        "deployments": [
            {"version": "17", "status": "successful"},
            {"version": "16", "status": "successful"},
        ],
    }

print(get_model_metrics("fraud-model"))


## 3. Describe those functions to Claude

This is the key boundary. We give Claude **schemas and descriptions**, not Python execution access.

In [ ]:
tools = [
    {
        "name": "get_model_metrics",
        "description": "Get production metrics for an ML model.",
        "input_schema": {
            "type": "object",
            "properties": {"model_name": {"type": "string"}},
            "required": ["model_name"],
        },
    },
    {
        "name": "get_kubernetes_status",
        "description": "Get Kubernetes deployment status for an ML model.",
        "input_schema": {
            "type": "object",
            "properties": {"model_name": {"type": "string"}},
            "required": ["model_name"],
        },
    },
    {
        "name": "get_recent_deployments",
        "description": "Get recent deployment history for an ML model.",
        "input_schema": {
            "type": "object",
            "properties": {"model_name": {"type": "string"}},
            "required": ["model_name"],
        },
    },
]

print(json.dumps(tools, indent=2))


## 4. Ask Claude to use the tools

Run this cell and inspect the raw response. Instead of normal text, Claude may return a `tool_use` block.

In [ ]:
messages = [{
    "role": "user",
    "content": "Check the fraud-model. What are its accuracy, latency, and Kubernetes health?"
}]

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    tools=tools,
    messages=messages,
)

for block in response.content:
    print("TYPE:", block.type)
    print(block)
    print("-" * 60)


### What you should notice

Claude can return something conceptually like:

```text
tool_use
name = get_model_metrics
input = {"model_name": "fraud-model"}
```

That is **a request**, not execution. Python still has to decide whether and how to execute it.

## 5. Build the tool dispatcher

This is your application's security/control boundary. In production, this is where you can validate arguments, authorize actions, log calls, and reject dangerous tools.

In [ ]:
FUNCTIONS = {
    "get_model_metrics": get_model_metrics,
    "get_kubernetes_status": get_kubernetes_status,
    "get_recent_deployments": get_recent_deployments,
}


def execute_tool(name, arguments):
    print(f"\nPYTHON TOOL CALL -> {name}")
    print("arguments ->", arguments)

    if name not in FUNCTIONS:
        return {"error": f"Tool '{name}' is not allowed"}

    try:
        result = FUNCTIONS[name](**arguments)
        print("result ->", result)
        return result
    except Exception as exc:
        return {"error": str(exc)}


## 6. Manually complete one tool call

Now we connect the two sides: Claude's `tool_use` request becomes an actual Python function call, then the result goes back to Claude.

In [ ]:
messages.append({"role": "assistant", "content": response.content})

tool_results = []
for block in response.content:
    if block.type == "tool_use":
        result = execute_tool(block.name, block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(result),
        })

messages.append({"role": "user", "content": tool_results})

final_response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=messages,
)

for block in final_response.content:
    if block.type == "text":
        print("\nCLAUDE FINAL ANSWER:\n")
        print(block.text)


## 7. Now automate the agent loop

This is essentially what an SDK helper such as a tool runner is doing for you: repeat **Claude → tool request → Python → tool result → Claude** until Claude finishes.

In [ ]:
def run_agent(user_prompt, max_turns=8):
    messages = [{"role": "user", "content": user_prompt}]

    for turn in range(max_turns):
        print(f"\n========== TURN {turn + 1} ==========")

        response = client.messages.create(
            model=MODEL,
            max_tokens=1000,
            tools=tools,
            messages=messages,
        )

        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "text":
                print("CLAUDE:", block.text)

            elif block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })

        if not tool_results:
            return response

        messages.append({"role": "user", "content": tool_results})

    raise RuntimeError("Agent exceeded maximum turns")


## 8. Try it yourself

Change the prompt. Claude chooses the tools and arguments.

In [ ]:
run_agent("""
Investigate fraud-model.
Give me its accuracy, latency, current Kubernetes health,
and recent deployment history. Summarize anything unusual.
""")


## 9. Try a different model

The important point is that your Python tool layer doesn't change. Only the LLM changes its reasoning/selection behavior.

In [ ]:
run_agent("""
Compare fraud-model and recommendation-model.
Which one has lower latency and what are their accuracy values?
""")


## 10. Add a deliberately unsafe tool

This demonstrates why the dispatcher matters. An LLM should not automatically receive unrestricted Python, shell, Kubernetes, or cloud credentials.

In [ ]:
def delete_production_model(model_name: str):
    return {"blocked": True, "message": f"Deletion of {model_name} requires human approval."}

# Deliberately NOT added to FUNCTIONS.
print("Dangerous functions should require explicit authorization and policy checks.")


## 11. The architecture you just built

```text
                    USER
                      |
                      v
                  CLAUDE API
                      |
             chooses a tool + arguments
                      |
                      v
              PYTHON DISPATCHER
                 /     |      \
                v      v       v
             MLflow  K8s   Prometheus
                |      |       |
                +------+-------+
                       |
                       v
                 tool_result
                       |
                       v
                    CLAUDE
                       |
                       v
                 FINAL ANSWER
```

### Next lab: MCP

The next step is to move the tool definitions/implementations behind an **MCP server**, then connect Claude to that server. The useful comparison is not 'tool calling versus MCP' but:

**local Python tools → tool runner → MCP server → reusable AI tool infrastructure.**
